# ParkCast Vision — Week 4: CLIP 기반 VLM 질의

`parkcast/vlm.py`의 `ParkingVLM` 클래스(CLIP 기반 zero-shot 이미지-텍스트 매칭)를
실제로 실행하는 노트북.

## 데모 구성
1. **Zero-shot 차종 분류**: Week 1 bbox 모델로 검출한 점유 칸을 crop → CLIP으로
   sedan/SUV/truck/motorcycle 분류
2. **자연어 질의**: 이미지 전체를 자유 텍스트와 비교(CLIP은 질문에 "대답"하는 게 아니라
   후보 문장들과의 유사도 순위를 매기는 것 — 아래 3번 셀 참조)
3. **Seg + VLM 결합**: Week 3 seg 모델의 폴리곤 마스크로 배경을 지운 뒤 crop → bbox
   crop과 비교해서 "배경 노이즈 없이 분류"가 실제로 더 나은지 확인


## 0. 환경 확인 + 패키지 설치

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')


In [ ]:
!pip install -q ultralytics transformers accelerate


## 1. 저장소 준비 (parkcast 패키지 import)

In [ ]:
import os, sys

REPO_URL = 'https://github.com/kth020829-cell/Competition.git'
REPO_DIR = '/content/Competition'
PARKCAST_DIR = f'{REPO_DIR}/2025 국토교통 데이터활용 경진대회/parkcast'

if not os.path.exists(PARKCAST_DIR):
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"
else:
    print('리포지토리 이미 존재:', REPO_DIR)

sys.path.insert(0, PARKCAST_DIR)
import parkcast
print('parkcast 로드 위치:', parkcast.__file__)


## 2. Drive 마운트 + 경로 설정

가중치 파일명이 노트북마다 조금씩 다르게 저장돼 있었음(Week1은 학습 코드가 노트북에
인라인으로 있어서 `yolov8n_pklot_week1_best.pt`, Week3는 `scripts/train.py`를 통해
`configs/segment.yaml`의 `run_name` 그대로 `yolov8n_seg_pklot_v1_best.pt`로 저장됨).
아래 `resolve_weights`가 후보 이름들을 순서대로 찾아줌 — 전부 실패하면 `MODEL_DIR`
안의 실제 파일 목록을 보여주니 그중 맞는 이름으로 `DETECT_WEIGHTS`/`SEG_WEIGHTS`를
직접 지정하면 됨.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/ParkCast'
MODEL_DIR = f'{PROJECT_ROOT}/models'


def resolve_weights(candidates, label):
    for name in candidates:
        path = f'{MODEL_DIR}/{name}'
        if os.path.exists(path):
            print(f'{label} 가중치: {path}')
            return path
    existing = os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else []
    raise FileNotFoundError(
        f'{label} 가중치를 후보 이름 {candidates} 중에서 못 찾음.\n'
        f'{MODEL_DIR} 안의 실제 파일: {existing}\n'
        f'위 목록에서 맞는 이름을 candidates에 추가하거나 직접 지정하면 됨.'
    )


# Week 1 (bbox detection, 전체 데이터)
DETECT_WEIGHTS = resolve_weights(
    ['yolov8n_pklot_week1_best.pt', 'yolov8n_pklot_v1_best.pt'], 'Week1 detect'
)

# Week 3 (SAM 자동 라벨링 + YOLOv8n-seg, 서브셋)
# configs/segment.yaml의 run_name 기준으로는 yolov8n_seg_pklot_v1_best.pt인데,
# 다른 이름으로 저장했을 가능성도 있어 후보에 둘 다 넣어둠 — 실제 존재하는 쪽이 자동으로 잡힘.
SEG_WEIGHTS = resolve_weights(
    ['yolov8n_seg_pklot_v1_best.pt', 'yolov8n_seg_sam_best.pt'], 'Week3 seg'
)


## 3. 데모용 이미지 준비

Week 1에서 이미 변환해둔 YOLO(detect) 데이터셋의 test 이미지를 그대로 씀. 로컬(`/content`)에
없으면 Drive 백업에서 복구(Week1/Week2 노트북과 동일한 방식).

Demo 3(seg)도 이 이미지들을 그대로 씀 — seg 모델 추론은 라벨이 필요 없고 이미지 파일만
있으면 되므로, `pklot_yolo_seg/`(SAM 자동 라벨링 결과물)를 따로 만들 필요 없음.


In [ ]:
import glob

DATA_ROOT = '/content'
YOLO_ROOT = '/content/pklot_yolo'

if not os.path.exists(f'{YOLO_ROOT}/test/images'):
    backup = f'{PROJECT_ROOT}/pklot-dataset.zip'
    assert os.path.exists(backup), f'{backup} 없음 — Week1 노트북을 먼저 실행해 데이터를 받아둬야 함.'
    print('PKLot raw 데이터 복구 중...')
    !cp "{backup}" /content/
    !cd /content && unzip -q pklot-dataset.zip
    from parkcast.data import coco_to_yolo
    coco_to_yolo(DATA_ROOT, YOLO_ROOT)

test_images = sorted(glob.glob(f'{YOLO_ROOT}/test/images/*.jpg'))
print(f'test 이미지 {len(test_images)}장 (Demo 1/2/3 공용 — 라벨 없이 이미지만 있으면 됨)')

# 차가 많이 찍힌(점유 칸이 많은) 이미지가 데모에 유리하므로, 앞에서 두 장을 미리 골라둠
sample_image = test_images[0]
seg_sample_image = test_images[1] if len(test_images) > 1 else test_images[0]
print('Demo 1/2용 샘플 이미지:', sample_image)
print('Demo 3용 샘플 이미지:  ', seg_sample_image)


## 4. CLIP 모델 로드 (ParkingVLM)

In [ ]:
from parkcast.vlm import ParkingVLM, DEFAULT_VEHICLE_LABELS, DEFAULT_SCENE_LABELS

try:
    vlm = ParkingVLM()  # 기본값: openai/clip-vit-base-patch32
    print('CLIP 로드 완료. device =', vlm.device)
except Exception as e:
    print('CLIP 로드 실패:', repr(e))
    print('GPU 메모리 부족이면 재시작 후 재시도, 그 외 에러면 transformers 버전 확인 필요.')
    raise


## Demo 1 — Zero-shot 차종 분류

Week 1 bbox 모델로 점유 칸을 검출하고, 각 칸을 crop해서 CLIP에 넣어 차종(sedan/SUV/
truck/motorcycle)을 분류함. `parkcast.vlm.ParkingVLM.classify_boxes`가 이 작업을 함.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from parkcast.inference import OccupancyPredictor
from parkcast.utils import is_empty_class

detect_predictor = OccupancyPredictor(DETECT_WEIGHTS)
result = detect_predictor.predict(sample_image, conf=0.4)

img_bgr = cv2.imread(sample_image)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

occ_idx = [i for i, n in enumerate(result.class_names) if not is_empty_class(n)]
occ_boxes = result.boxes_xyxy[occ_idx][:6]  # 너무 많으면 시각화가 복잡하니 최대 6개만
print(f'점유 칸 {len(occ_idx)}개 검출 — 상위 {len(occ_boxes)}개를 CLIP으로 차종 분류')

vehicle_results = vlm.classify_boxes(img_rgb, occ_boxes, DEFAULT_VEHICLE_LABELS)

fig, axes = plt.subplots(1, len(occ_boxes), figsize=(4 * len(occ_boxes), 4))
axes = axes if hasattr(axes, '__len__') else [axes]
for ax, box, (label, prob) in zip(axes, occ_boxes, vehicle_results):
    x1, y1, x2, y2 = box.astype(int)
    crop = img_rgb[max(0, y1):y2, max(0, x1):x2]
    ax.imshow(crop)
    ax.set_title(f'{label}\n{prob*100:.1f}%', fontsize=10)
    ax.axis('off')
plt.suptitle('Demo 1 — Zero-shot 차종 분류 (bbox crop)')
plt.tight_layout()
plt.show()


## Demo 2 — 자연어 질의

이미지 전체를 자유 텍스트 질의 + 기본 후보 문장들과 비교함. **CLIP은 질문에 "대답"하는
VQA 모델이 아니라 이미지-텍스트 임베딩 유사도 모델**이므로, 아래 결과는 "이 질의가 후보들
사이에서 상대적으로 얼마나 그럴듯한가"로 읽어야 함(절대적인 정답이 아님).


In [ ]:
queries = [
    'a truck blocking two spaces',
    'a parking lot with tree shadows on the ground',
    'an almost empty parking lot',
]

for q in queries:
    ranked = vlm.classify(img_rgb, DEFAULT_SCENE_LABELS, extra_query=q)
    print(f'질의: "{q}"')
    for label, prob in ranked[:3]:
        marker = '  <-- 이 질의' if label == q else ''
        print(f'  {label}: {prob*100:.1f}%{marker}')
    print()


## Demo 3 — Seg + VLM 결합 파이프라인 (배경 제거 후 분류)

Week 3 seg 모델은 인스턴스별 polygon 마스크(`result.masks_xy`)를 주는데, `vlm.py`의
`classify_boxes`는 bbox crop만 지원하고 마스크 기반 배경 제거는 없음(vlm.py를 건드리지
않고 이 셀에서 직접 구현함). polygon 바깥을 흰색으로 채워서 옆 칸/배경 차량이 섞여
들어가지 않게 한 뒤 CLIP에 넣어, 같은 인스턴스를 bbox crop으로 분류했을 때와 비교함.


In [ ]:
def mask_crop(image_rgb, polygon_xy, bg_color=(255, 255, 255)):
    """polygon 바깥을 bg_color로 지우고 polygon의 bounding box로 잘라냄.

    bbox crop과 달리 옆 칸 차량·배경이 섞이지 않음 — "배경 노이즈 없이 분류"의 핵심.
    """
    mask = np.zeros(image_rgb.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [polygon_xy.astype(np.int32)], 255)
    masked = image_rgb.copy()
    masked[mask == 0] = bg_color
    x, y, w, h = cv2.boundingRect(polygon_xy.astype(np.int32))
    return masked[y:y + h, x:x + w]


seg_predictor = OccupancyPredictor(SEG_WEIGHTS)
seg_result = seg_predictor.predict(seg_sample_image, conf=0.4)
assert seg_result.has_masks, (
    f'{SEG_WEIGHTS}가 seg 모델이 아닌 것 같음(masks_xy가 없음) — SEG_WEIGHTS 경로 확인 필요'
)

seg_img_bgr = cv2.imread(seg_sample_image)
seg_img_rgb = cv2.cvtColor(seg_img_bgr, cv2.COLOR_BGR2RGB)

occ_idx_seg = [i for i, n in enumerate(seg_result.class_names) if not is_empty_class(n)][:4]
print(f'점유 칸 {len(occ_idx_seg)}개에 대해 bbox crop vs mask crop 비교')

fig, axes = plt.subplots(2, len(occ_idx_seg), figsize=(4 * len(occ_idx_seg), 8), squeeze=False)
for col, i in enumerate(occ_idx_seg):
    box = seg_result.boxes_xyxy[i].astype(int)
    bbox_crop = seg_img_rgb[max(0, box[1]):box[3], max(0, box[0]):box[2]]
    seg_crop = mask_crop(seg_img_rgb, seg_result.masks_xy[i])

    bbox_label, bbox_prob = vlm.classify(bbox_crop, DEFAULT_VEHICLE_LABELS)[0]
    seg_label, seg_prob = vlm.classify(seg_crop, DEFAULT_VEHICLE_LABELS)[0]

    axes[0][col].imshow(bbox_crop)
    axes[0][col].set_title(f'bbox: {bbox_label}\n{bbox_prob*100:.1f}%', fontsize=9)
    axes[0][col].axis('off')

    axes[1][col].imshow(seg_crop)
    axes[1][col].set_title(f'seg(배경제거): {seg_label}\n{seg_prob*100:.1f}%', fontsize=9)
    axes[1][col].axis('off')

    print(f'인스턴스 {i}: bbox → {bbox_label} ({bbox_prob*100:.1f}%)  |  '
          f'seg(배경제거) → {seg_label} ({seg_prob*100:.1f}%)')

axes[0][0].set_ylabel('bbox crop', fontsize=11)
axes[1][0].set_ylabel('mask crop (배경 제거)', fontsize=11)
plt.suptitle('Demo 3 — bbox crop vs seg mask crop, CLIP 차종 분류 비교')
plt.tight_layout()
plt.show()


## 5. Gradio 데모 실행

`app/gradio_demo.py`는 위 세 데모를 웹 UI로 통합한 것 — Detection 탭(점유율 검출) +
VLM 질의 탭(자연어 질의, 차종 추정). seg 모델을 넘기면 VLM 탭이 자동으로
`classify_masks`(배경 제거 crop)를 쓰고, detect 모델을 넘기면 `classify_boxes`(bbox
crop)로 자동 전환됨(`app/gradio_demo.py`의 `vlm_vehicle_fn` 참조).

아래 셀은 그대로 실행하면 `*.gradio.live` 공개 링크가 출력됨 — 셀을 중단하기 전까지 살아있음.


In [ ]:
# SEG_WEIGHTS로 실행하면 Detection 탭의 마스크 오버레이 + VLM 탭의 배경 제거 분류를
# 한 번에 보여줄 수 있어서 데모 효과가 더 큼. DETECT_WEIGHTS로 바꾸고 싶으면 아래 한 줄만 교체.
DEMO_WEIGHTS = SEG_WEIGHTS

!python "{PARKCAST_DIR}/app/gradio_demo.py" --weights "{DEMO_WEIGHTS}" --share


## 6. HuggingFace Spaces 배포

`app/hf_space/`에 이미 준비해둔 Space 진입점(`app.py`)을 실제 Space에 업로드함.
`huggingface_hub`의 `upload_folder`/`upload_file`로 git 없이 바로 올릴 수 있음(Colab
안에서 완결). **SPACE_ID를 실제 자기 Space로 바꾸고, Space Settings에서 SDK가
"Gradio"인지 먼저 확인할 것** — 다른 SDK(Docker/Static 등)면 `app.py`가 인식되지 않음.


In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import login, HfApi

login()  # 토큰 입력창이 뜸 (huggingface.co/settings/tokens 에서 write 권한 토큰 발급)

SPACE_ID = "your-username/your-space-name"  # ← 실제 Space로 바꿀 것

api = HfApi()

# 1) Space 코드 업로드 (app.py, requirements.txt, README.md)
api.upload_folder(
    folder_path=f"{PARKCAST_DIR}/app/hf_space",
    repo_id=SPACE_ID,
    repo_type="space",
)

# 2) app.py가 import하는 parkcast 패키지 업로드
api.upload_folder(
    folder_path=f"{PARKCAST_DIR}/parkcast",
    repo_id=SPACE_ID,
    repo_type="space",
    path_in_repo="parkcast",
)

# 3) 가중치 업로드 — app.py의 _resolve_weights()가 정확히 이 경로(models/best.pt)를 찾음
api.upload_file(
    path_or_fileobj=DEMO_WEIGHTS,
    repo_id=SPACE_ID,
    repo_type="space",
    path_in_repo="models/best.pt",
)

print(f'배포 완료: https://huggingface.co/spaces/{SPACE_ID}')
print('Space가 재빌드되는 데 1~2분 걸림 — 위 링크에서 로그 확인 가능')
